In [1]:
import os 
from quad_race_env import *
from randomization import *
from stable_baselines3 import PPO
from quadcopter_animation import animation
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_log(log_dir):
    event_acc = EventAccumulator(log_dir)
    event_acc.Reload()
    tags = event_acc.Tags()['scalars']
    
    # print(tags)
    # raise Exception('stop')
    log = {'step': [], 'ep_rew_mean': []}
    
    for scalar_event in event_acc.Scalars('rollout/ep_rew_mean'):
        log['step'].append(scalar_event.step)
        log['ep_rew_mean'].append(scalar_event.value)
        
    log = {key: np.array(values) for key, values in log.items()}
    # sub sample in steps of 1e6
    indices = log['step'] % 1e6 == 0
    log = {key: values[indices] for key, values in log.items()}
    # get the highest value and corresponding step
    max_index = np.argmax(log['ep_rew_mean'])
    log['max_ep_rew_mean'] = log['ep_rew_mean'][max_index]
    log['max_step'] = log['step'][max_index]
    return log
    
def get_best_model(log_dir):
    log = get_log(log_dir)
    # print('highest mean episode reward:', log['max_ep_rew_mean'], 'at step:', log['max_step'])
    model_dir = log_dir
    model_dir = model_dir.replace('logs', 'models')
    model_dir = model_dir[:-2]
    path_best_model = model_dir + f'/{log["max_step"]}.zip'
    print('best model:', path_best_model, 'mean_ep_rew:', log['max_ep_rew_mean'])
    return path_best_model, log['max_ep_rew_mean']

env = Quadcopter3DGates(
    num_envs=1,
    randomization=randomization_fixed_params_dummy,
    initialize_at_random_gates=False,
    initialize_on_ground=True,
    pause_if_collision=False,
)

# get latest model from model/ground_exp
def get_latest_model(path):
    models = os.listdir(path)
    # all files end with {number}.zip, get the highest number
    models = sorted(models, key=lambda x: int(x.split('.')[0].split('_')[-1]))
    print(models[-1])
    model = PPO.load(path + '/' + models[-1])
    return model

#  ANIMATION FUNCTION
action_list = []
reward_list = []
def animate_policy(model, env, **kwargs):
    env.reset()
    def run():
        actions, _ = model.predict(env.states, deterministic=False)
        action_list.append(actions)
        states, rewards, dones, infos = env.step(actions)
        reward_list.append(rewards)
        # print('speed = ', np.linalg.norm(states[0][3:6]))
        # print(infos)
        return env.render()
    animation.view(run, gate_pos=env.gate_pos, gate_yaw=env.gate_yaw, **kwargs)

# model, _ = get_best_model('logs/ground_exp/test1_0')
# model, _ = get_best_model('logs/perception_exp/test2_0')
model, _ = get_best_model('logs/perception_exp/cool_split6_0')
model = PPO.load(model)
animate_policy(model, env)

python version: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
stable_baselines3 version: 2.3.2
torch version: 2.4.1
cuda available: False
cuda version: 12.4
cudnn version: 90100
device: cpu


/home/robinferede/miniconda3/envs/quad/lib/python3.10/site-packages/torch/cuda/__init__.py:128: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /opt/conda/conda-bld/pytorch_1724789122112/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


test
<function _lambdifygenerated at 0x70d14019c940>
10 10


/home/robinferede/miniconda3/envs/quad/lib/python3.10/site-packages/stable_baselines3/common/vec_env/base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


best model: models/perception_exp/cool_split6/379000000.zip mean_ep_rew: 38.599449157714844


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/robinferede/miniconda3/envs/quad/lib/python3.10/site-packages/cv2/qt/plugins"


unpaused
paused
unpaused
paused
unpaused
